Baseline data extraction, delay calculation, fairness setup, and capacity penalty construction

This block prepares the full “before control” baseline using the PeMS 5-minute dataset for the
selected I-405 southbound corridor. The goal is to extract the observed freeway and ramp data,
compute benchmark mainline delay, build local-road delay scenarios, define the fairness penalty,
and construct the three capacity-related penalty terms that will later be reused inside the CTM
and ADMM model.

--------------------------------------------------------------------------------------------------
Part 1. Load the raw PeMS dataset and extract only the corridor stations we care about

The code first loads:

* the station metadata file, which helps verify detector IDs and locations, and
* the 5-minute station dataset, which contains timestamped flow, occupancy, and speed values.

Then it filters the large district-wide dataset so that only the selected station IDs remain.
These IDs include:

* mainline detectors
* on-ramp detectors
* off-ramp detectors

Two time windows are extracted:

1. a low-congestion / early-morning window, used to estimate free-flow speed
2. the 08:00 morning-congestion window, used to measure observed corridor conditions

After filtering, only the important columns are kept:

* timestamp
* station_id
* station_type
* total_flow
* avg_occupancy
* avg_speed

This reduced dataset becomes the starting point for all later calculations.

--------------------------------------------------------------------------------------------------
Part 2. Mainline delay benchmark using observed PeMS data

This section computes a flow-based mainline-delay baseline, meaning it uses observed flow and
observed travel time directly from the dataset, before any CTM or ADMM control is applied.

The computation is done in several steps.

Step 1. Estimate free-flow speed for each mainline station

For each selected mainline detector, the code computes the median speed during the chosen
low-congestion time window.

This gives a station-level free-flow speed estimate:

$$\tilde v_s=\text{median of overnight speed samples at station } s$$

The reason for using the median is to reduce the effect of occasional noisy speed values.

Step 2. Convert station free-flow speeds into segment free-flow speeds

Each corridor segment lies between two consecutive mainline stations.
The free-flow speed of a segment is taken as the average of the upstream and downstream station
free-flow speeds:

$$v_i^{ff}=\frac{\tilde v_{up}+\tilde v_{down}}{2}$$

Step 3. Compute free-flow travel time for each segment

Once free-flow speed is known, free-flow travel time is:

$$TT_i^{ff}=\frac{L_i}{v_i^{ff}}$$

where:

* $L_i$ = segment length in miles
* $v_i^{ff}$ = free-flow speed in mph

The code stores both:

* travel time in hours
* travel time in minutes

Step 4. Compute observed segment speed at 08:00

For the morning peak dataset, the code gets the actual observed speed at each mainline station.
Then, for each segment, it averages the upstream and downstream observed speeds:

$$v_i^{obs}=\frac{v_{up}^{obs}+v_{down}^{obs}}{2}$$

Step 5. Compute observed travel time for each segment

Using the observed segment speed:

$$TT_i^{obs}=\frac{L_i}{v_i^{obs}}$$

This gives the actual travel time experienced during the selected congested 5-minute interval.

Step 6. Compute delay per vehicle

The segment delay per vehicle is the difference between observed travel time and free-flow travel time:

$$d_i=TT_i^{obs}-TT_i^{ff}$$

This is computed in both hours and minutes.

Step 7. Compute total mainline delay for each segment

The code then multiplies per-vehicle delay by the observed segment flow:

$$D_{M,i}=q_i \cdot d_i$$

where:

* $q_i$ = observed segment flow during the 5-minute interval
* $d_i$ = per-vehicle delay

This gives total mainline delay in:

* vehicle-minutes
* vehicle-hours

This result is the observed, no-control benchmark for the freeway mainline.

NOTE:
This is a flow-based benchmark. Later, the CTM model will use a state-based mainline delay
formulation, which includes stored vehicles inside the cells and is better suited for dynamic
simulation and optimization.

--------------------------------------------------------------------------------------------------
Part 3. Local-road delay setup using ramp queue scenarios

The raw dataset gives observed ramp discharge flow, but it does not directly provide the full
ramp queue state at the beginning of the interval. Because of that, this section uses scenario-
based assumptions to understand how local-road delay could behave under different queue and
arrival conditions.

Step 1. Extract 08:00 on-ramp flow values

The code filters the dataset to keep only the on-ramp detectors at the selected time.
These ramp flows are used as:

* discharged ramp flow, $u_t$

Step 2. Define the ramp queue update equation

Ramp state evolves using:

$$R_t = R_{t-1}+a_t-u_t$$

where:

* $R_{t-1}$ = queue at the beginning of the interval
* $a_t$ = newly arriving vehicles to the ramp during the interval
* $u_t$ = discharged vehicles released from the ramp

Step 3. Define local-road delay

Local delay is measured using the trapezoidal approximation:

$$D_{L,i}(t)=\left(\frac{R_{t-1}+R_t}{2}\right)\Delta T$$

where:

* $\Delta T = 5$ minutes in this baseline calculation block

Step 4. Test multiple queue scenarios

Because the initial queue is unknown, the code tests several possible values:

* $R_{t-1}\in \{0,10,20,30\}$

Step 5. Test multiple arrival assumptions

Because actual ramp arrivals are unknown, the code tests:

* $a_t = u_t$
* $a_t = 1.5u_t$
* $a_t = 2u_t$

For every combination of:

* initial queue assumption
* arrival assumption

the code computes:

* next ramp queue
* local delay for each ramp
* total local delay across all ramps

This gives a scenario-based local-road benchmark and shows how sensitive local delay is to the
assumed arrival and initial queue values.

--------------------------------------------------------------------------------------------------
Part 4. Fairness penalty setup using ramp stress

The purpose of the fairness term is to discourage a control policy from overloading one ramp much
more than the others.

Step 1. Compute maximum ramp storage

Ramp maximum queue storage is estimated from physical ramp length and number of lanes:

$$R_{max}=\frac{\text{ramp length}\times \text{number of lanes}}{\text{average vehicle length}}$$

where:

* average vehicle length is assumed to be 25 ft

This gives the approximate maximum number of vehicles each ramp can physically store.

Step 2. Compute ramp stress index

Ramp stress is defined as:

$$\phi_i=\frac{R_i}{R_{max,i}}$$

Interpretation:

* $\phi_i=0$ means empty ramp
* $\phi_i=0.5$ means ramp is half full
* $\phi_i=1$ means ramp is full
* $\phi_i>1$ means spillback is occurring

Step 3. Cap stress at 1 for fairness

Since fairness is meant to compare relative burden rather than reward extreme overflow, stress is
capped:

$$\phi_i^{cap}=\min(\phi_i,1)$$

This means any ramp already beyond full storage is treated as fully stressed for fairness
comparison.

Step 4. Compute pairwise fairness penalty

The fairness penalty compares every pair of ramps:

$$L_{fair}=\gamma \sum_{i<j}(\phi_i^{cap}-\phi_j^{cap})^2$$

where:

* $\gamma$ = fairness multiplier

The code evaluates this penalty for:

* several ramp queue assumptions
* several arrival assumptions
* several fairness multipliers

This shows how fairness changes when ramp congestion becomes more uneven.

Interpretation:

* equal stress across ramps -> low fairness penalty
* one or more ramps much more stressed than others -> high fairness penalty

--------------------------------------------------------------------------------------------------
Part 5. Doorway capacity construction

This section builds the first capacity-related term: doorway capacity.

Doorway capacity represents the maximum number of vehicles that can enter / pass through a segment
during one interval before flow-processing limits are exceeded.

Step 1. Start from known station capacities

Some stations already have known doorway capacities in vehicles per 5-minute interval.

Step 2. Convert known capacities into per-lane values

For each station with known capacity:

$$\text{per-lane doorway capacity}=\frac{\text{known total capacity}}{\text{number of lanes}}$$

Step 3. Average across stations

The code then computes an average per-lane doorway capacity using the known stations.

Step 4. Estimate missing capacities

For stations with missing doorway capacity, the code estimates:

$$C_i=(\text{average per-lane doorway capacity})\times(\text{lane count})$$

This produces a full doorway-capacity table for all corridor stations / segments.

--------------------------------------------------------------------------------------------------
Part 6. Doorway capacity penalty

The doorway capacity penalty activates when total inflow into a segment exceeds its doorway
capacity.

Formula:

$$L_{\text{doorway},i}=\lambda_1 \max(0,Q_{in,i}+u_i-C_i)^2$$

where:

* $Q_{in,i}$ = mainline inflow to segment i
* $u_i$ = total on-ramp inflow entering segment i
* $C_i$ = doorway capacity of segment i
* $\lambda_1$ = doorway penalty multiplier

The code tests this penalty using:

* actual observed flow values
* synthetic larger flow values

The synthetic case is used as a sanity check to make sure the penalty activates correctly when
inflow exceeds capacity.

--------------------------------------------------------------------------------------------------
Part 7. Physical storage capacity

Physical capacity is the hard upper limit on how many vehicles can physically fit inside a
segment.

Formula:

$$N_{max,i}=k_j \cdot L_i \cdot n_i$$

where:

* $k_j$ = jam density in veh/mi/ln
* $L_i$ = segment length
* $n_i$ = number of lanes

This gives the maximum segment storage before full jam density is reached.

--------------------------------------------------------------------------------------------------
Part 8. Initial and final mainline state for the segment-based baseline

To evaluate storage-related penalties, the code needs an estimate of how many vehicles are already
inside each segment at the beginning and end of the 5-minute interval.

Step 1. Compute station density

Station density is estimated using observed flow and speed:

$$k=\frac{q^{hr}}{v}$$

Since PeMS flow is reported per 5 minutes, it is converted to hourly flow first:

$$q^{hr}=12\cdot q^{5min}$$

Step 2. Compute average segment density

For each segment, density is averaged between the upstream and downstream stations.

Step 3. Compute initial vehicles in segment

$$X_{initial,i}=k_{avg,i}\cdot L_i$$

Step 4. Compute final vehicles in segment using conservation

$$X_{final,i}=X_{initial,i}+Q_{in,i}+u_i-Q_{out,i}-f_i$$

where:

* $Q_{in,i}$ = mainline inflow
* $u_i$ = on-ramp inflow
* $Q_{out,i}$ = mainline outflow
* $f_i$ = off-ramp outflow

This gives an approximate segment-level state update using the observed 5-minute data.

--------------------------------------------------------------------------------------------------
Part 9. Physical-capacity penalty

This penalty activates only when the final segment occupancy exceeds hard physical storage.

Formula:

$$L_{\text{physical},i}=\lambda_3 \max(0,X_{final,i}-N_{max,i})^2$$

where:

* $X_{final,i}$ = estimated vehicles in segment i at the end of the interval
* $N_{max,i}$ = physical storage limit
* $\lambda_3$ = physical-capacity multiplier

The code first checks this using actual data.
Since actual occupancies are below hard physical storage, the penalty becomes zero.

Then the code uses synthetic oversized states as a sanity check to verify that the penalty logic
works correctly when overflow exists.

--------------------------------------------------------------------------------------------------
Part 10. Safe-threshold capacity

Safe threshold is a soft storage limit placed below the hard physical maximum.
It is used to penalize unstable congestion before the segment reaches full jam density.

Formula:

$$X_{safe,i}=\eta N_{max,i}$$

where:

* $\eta$ = threshold multiplier, usually between 0 and 1
* $N_{max,i}$ = physical capacity

The code tests multiple values of:

* $\eta = 0.3, 0.5, 0.7$

This allows inspection of how aggressive or relaxed the soft-threshold should be.

--------------------------------------------------------------------------------------------------
Part 11. Safe-threshold penalty

This penalty activates when segment occupancy exceeds the soft threshold:

$$L_{\text{safe},i}=\lambda_2 \max(0,X_{final,i}-X_{safe,i})^2$$

where:

* $X_{final,i}$ = estimated final segment occupancy
* $X_{safe,i}$ = soft threshold
* $\lambda_2$ = safe-threshold multiplier

The code evaluates this penalty for:

* actual segment states
* synthetic oversized states
* several choices of eta
* several values of lambda_2

This shows:

* whether the actual data crosses the soft threshold
* how sensitive the penalty is to the threshold choice
* how strongly the penalty scales with lambda_2

--------------------------------------------------------------------------------------------------
Final interpretation of this whole block

This code block builds the full baseline and penalty structure needed before moving to CTM and
ADMM.

In summary, it does five major jobs:

1. extracts observed mainline, ramp, and off-ramp data from PeMS
2. computes the observed no-control mainline delay benchmark
3. explores local-road delay and fairness using ramp queue scenarios
4. constructs the three mainline capacity terms:
   * doorway capacity
   * safe threshold
   * physical capacity
5. verifies all penalty equations using both actual and synthetic test values

This section is important because it creates the numerical baseline, penalty definitions, and
calibration logic that are later reused when the corridor is rebuilt as a CTM network and
optimized with ADMM-based ramp control.

In [172]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import os
%matplotlib inline


In [173]:
# 1. Read station metadata and display selected stations clearly

import pandas as pd

ids_to_keep_str = {str(x).strip() for x in ids_to_keep}

metadata_raw = pd.read_csv(
    "Station Metadata_district12.txt",
    sep=None,
    engine="python",
    header=None,
    dtype=str,
    skip_blank_lines=True
)

metadata_raw = metadata_raw.apply(lambda col: col.astype(str).str.strip())

selected_metadata_clean = pd.DataFrame({
    "station_id": metadata_raw[0],
    "station_name": metadata_raw[13],
    "station_type": metadata_raw[11],
    "freeway": metadata_raw[1],
    "direction": metadata_raw[2],
    "absolute_postmile": metadata_raw[7],
    "length": metadata_raw[10],
    "lanes": metadata_raw[12],
    "latitude": metadata_raw[8],
    "longitude": metadata_raw[9],
})

# Keep only selected stations
selected_metadata_clean["station_id"] = selected_metadata_clean["station_id"].astype(str).str.strip()

selected_metadata_clean = selected_metadata_clean[
    selected_metadata_clean["station_id"].isin(ids_to_keep_str)
].copy()

# Convert numeric columns
numeric_cols = [
    "station_id",
    "freeway",
    "absolute_postmile",
    "length",
    "lanes",
    "latitude",
    "longitude",
]

for col in numeric_cols:
    selected_metadata_clean[col] = pd.to_numeric(selected_metadata_clean[col], errors="coerce")

# Sort by corridor position
selected_metadata_clean = selected_metadata_clean.sort_values(
    ["absolute_postmile", "station_type"]
).reset_index(drop=True)

print("=== Selected Station Metadata ===")
print("Number of selected stations:", len(selected_metadata_clean))

display(selected_metadata_clean)

=== Selected Station Metadata ===
Number of selected stations: 15


,station_id,station_name,station_type,freeway,direction,absolute_postmile,length,lanes,latitude,longitude
0,1201419,RED HILL,ML,405,S,8.17,0.269,5,33.686517,-117.866474
1,1201465,BRISTOL 1,FR,405,S,9.31,NaN,2,33.687251,-117.886180
2,1201469,BRISTOL 1,ML,405,S,9.31,0.350,5,33.687251,-117.886180
3,1201460,BRISTOL 1,OR,405,S,9.31,NaN,1,33.687251,-117.886180
4,1201497,FAIRVIEW,ML,405,S,10.05,0.460,5,33.687480,-117.899035
5,1201490,FAIRVIEW,OR,405,S,10.07,NaN,1,33.687494,-117.899383
6,1201525,HARBOR 1,ML,405,S,10.97,0.610,6,33.687942,-117.915002
7,1201517,HARBOR 1,OR,405,S,10.97,NaN,1,33.687942,-117.915002
8,1201554,HARBOR 2,FR,405,S,11.27,NaN,3,33.689212,-117.919950
9,1201558,HARBOR 2,ML,405,S,11.27,0.480,5,33.689212,-117.919950


In [174]:

# 2. Load PeMS 5-minute station data
# Purpose:
# Load the full 5-minute detector dataset and create the 01:00–05:00
# free-flow window used to estimate free-flow speed.

# Load 5-minute PeMS station data
station_data = pd.read_csv("station_5min_district12.txt", header=None)

# Column 0 = timestamp
station_data[0] = pd.to_datetime(station_data[0])

# Final selected detector IDs:
# 7 mainline + 5 on-ramp + 3 off-ramp
ids_to_keep = [
    1201419, 1201469, 1201497, 1201525, 1201558, 1201589, 1201620,
    1201460, 1201490, 1201517, 1201548, 1201580,
    1201465, 1201554, 1201585
]

# Free-flow window
# This window is used only to estimate free-flow speed from low-congestion conditions.
start_time = "2026-01-08 01:00:00"
end_time   = "2026-01-08 05:00:00"

# Filter selected detectors during free-flow window
filtered_data_midnight = station_data[
    (station_data[1].isin(ids_to_keep)) &
    (station_data[0] >= pd.Timestamp(start_time)) &
    (station_data[0] <  pd.Timestamp(end_time))
].copy()

# Keep only useful columns
selected_data_midnight = filtered_data_midnight[[0, 1, 5, 9, 10, 11]].copy()

selected_data_midnight.columns = [
    "timestamp",
    "station_id",
    "station_type",
    "total_flow",
    "avg_occupancy",
    "avg_speed"
]

# Sort for readability
selected_data_midnight = selected_data_midnight.sort_values(
    ["timestamp", "station_id"]
).reset_index(drop=True)

print("Midnight / Free-Flow Window Data ")
print("Time window: 2026-01-08 01:00:00 to 2026-01-08 05:00:00")
print("Rows:", len(selected_data_midnight))
print("Unique timestamps:", selected_data_midnight["timestamp"].nunique())
print("Unique stations:", selected_data_midnight["station_id"].nunique())

print("\nFirst 20 rows:")
display(selected_data_midnight.head(20))

print("\nRows for station 1201419 (RED HILL):")
display(selected_data_midnight[selected_data_midnight["station_id"] == 1201419].head(20))

Midnight / Free-Flow Window Data 
Time window: 2026-01-08 01:00:00 to 2026-01-08 05:00:00
Rows: 720
Unique timestamps: 48
Unique stations: 15

First 20 rows:


,timestamp,station_id,station_type,total_flow,avg_occupancy,avg_speed
0,2026-01-08 01:00:00,1201419,ML,112.0,0.0345,47.3
1,2026-01-08 01:00:00,1201460,OR,0.0,0.0000,NaN
2,2026-01-08 01:00:00,1201465,FR,NaN,NaN,NaN
3,2026-01-08 01:00:00,1201469,ML,112.0,0.0257,61.4
4,2026-01-08 01:00:00,1201490,OR,9.0,0.0200,NaN
5,2026-01-08 01:00:00,1201497,ML,55.0,0.0049,69.8
6,2026-01-08 01:00:00,1201517,OR,3.0,0.0030,NaN
7,2026-01-08 01:00:00,1201525,ML,54.0,0.0097,64.5
8,2026-01-08 01:00:00,1201548,OR,6.0,0.0110,NaN
9,2026-01-08 01:00:00,1201554,FR,10.0,0.0033,NaN



Rows for station 1201419 (RED HILL):


,timestamp,station_id,station_type,total_flow,avg_occupancy,avg_speed
0,2026-01-08 01:00:00,1201419,ML,112.0,0.0345,47.3
15,2026-01-08 01:05:00,1201419,ML,174.0,0.0332,53.4
30,2026-01-08 01:10:00,1201419,ML,177.0,0.0398,53.7
45,2026-01-08 01:15:00,1201419,ML,217.0,0.0335,64.4
60,2026-01-08 01:20:00,1201419,ML,206.0,0.0344,68.4
75,2026-01-08 01:25:00,1201419,ML,194.0,0.0342,68.2
90,2026-01-08 01:30:00,1201419,ML,179.0,0.0312,68.0
105,2026-01-08 01:35:00,1201419,ML,155.0,0.0361,60.9
120,2026-01-08 01:40:00,1201419,ML,184.0,0.0401,57.7
135,2026-01-08 01:45:00,1201419,ML,188.0,0.0335,63.1


In [175]:
# 3. Load 08:00–09:00 benchmark window data
# Purpose:
# This is the actual peak-hour window used for the 8-cell / 30-sec CTM benchmark.
# PeMS gives 5-minute data, so 08:00–09:00 gives 12 intervals:
# 08:00, 08:05, ..., 08:55
# Each 5-min interval will later be divided into ten 30-sec CTM steps.

start_time = "2026-01-08 08:00:00"
end_time   = "2026-01-08 09:00:00"

filtered_data_morning = station_data[
    (station_data[1].isin(ids_to_keep)) &
    (station_data[0] >= pd.Timestamp(start_time)) &
    (station_data[0] <  pd.Timestamp(end_time))
].copy()

selected_data_morning = filtered_data_morning[[0, 1, 5, 9, 10, 11]].copy()

selected_data_morning.columns = [
    "timestamp",
    "station_id",
    "station_type",
    "total_flow",
    "avg_occupancy",
    "avg_speed"
]

selected_data_morning = selected_data_morning.sort_values(
    ["timestamp", "station_id"]
).reset_index(drop=True)

print(" Morning Benchmark Window Data ")
print("Time window: 2026-01-08 08:00:00 to 2026-01-08 09:00:00")
print("Rows:", len(selected_data_morning))
print("Unique timestamps:", selected_data_morning["timestamp"].nunique())
print("Unique stations:", selected_data_morning["station_id"].nunique())

print("\nFirst 30 rows:")
display(selected_data_morning.head(30))

print("\nRows for station 1201419 (RED HILL)")
display(selected_data_morning[selected_data_morning["station_id"] == 1201419])

 Morning Benchmark Window Data 
Time window: 2026-01-08 08:00:00 to 2026-01-08 09:00:00
Rows: 180
Unique timestamps: 12
Unique stations: 15

First 30 rows:


,timestamp,station_id,station_type,total_flow,avg_occupancy,avg_speed
0,2026-01-08 08:00:00,1201419,ML,645.0,0.1582,39.4
1,2026-01-08 08:00:00,1201460,OR,88.0,0.0840,NaN
2,2026-01-08 08:00:00,1201465,FR,NaN,NaN,NaN
3,2026-01-08 08:00:00,1201469,ML,681.0,0.1384,40.0
4,2026-01-08 08:00:00,1201490,OR,39.0,0.8700,NaN
5,2026-01-08 08:00:00,1201497,ML,622.0,0.0525,69.5
6,2026-01-08 08:00:00,1201517,OR,73.0,0.0980,NaN
7,2026-01-08 08:00:00,1201525,ML,737.0,0.1677,30.6
8,2026-01-08 08:00:00,1201548,OR,56.0,0.0980,NaN
9,2026-01-08 08:00:00,1201554,FR,82.0,0.0283,NaN



Rows for station 1201419 (RED HILL)


,timestamp,station_id,station_type,total_flow,avg_occupancy,avg_speed
0,2026-01-08 08:00:00,1201419,ML,645.0,0.1582,39.4
15,2026-01-08 08:05:00,1201419,ML,677.0,0.1536,39.8
30,2026-01-08 08:10:00,1201419,ML,703.0,0.1708,39.6
45,2026-01-08 08:15:00,1201419,ML,769.0,0.1629,43.6
60,2026-01-08 08:20:00,1201419,ML,726.0,0.1420,44.4
75,2026-01-08 08:25:00,1201419,ML,757.0,0.1616,47.5
90,2026-01-08 08:30:00,1201419,ML,772.0,0.1622,46.8
105,2026-01-08 08:35:00,1201419,ML,670.0,0.1642,44.0
120,2026-01-08 08:40:00,1201419,ML,715.0,0.1535,43.7
135,2026-01-08 08:45:00,1201419,ML,587.0,0.1551,38.0


In [176]:
# 5. Column reference for PeMS station_5min data

pems_column_reference = pd.DataFrame({
    "column_index": [0, 1, 5, 9, 10, 11],
    "column_name": [
        "timestamp",
        "station_id",
        "station_type",
        "total_flow",
        "avg_occupancy",
        "avg_speed"
    ]

})

display(pems_column_reference)

,column_index,column_name
0,0,timestamp
1,1,station_id
2,5,station_type
3,9,total_flow
4,10,avg_occupancy
5,11,avg_speed


## below is the starting point of the calculations using the dataset we collected


# mainline delay calculations


Free-flow speeds were estimated from the 01:00–05:00 low-congestion window. For each mainline detector, all 48 five-minute speed observations were collected, and the median speed was used as the station-level free-flow speed. The median was used instead of the mean to reduce sensitivity to abnormal detector readings or short disturbances.

In [177]:

# 5. Estimate free-flow speed for each mainline station
# Purpose:
# Use the low-congestion midnight window, 01:00–05:00,
# to estimate free-flow speed at each mainline detector.


mainline_ids = [
    1201419, 1201469, 1201497, 1201525, 1201558, 1201589, 1201620
]

mainline_midnight_data = selected_data_midnight[
    (selected_data_midnight["station_id"].isin(mainline_ids)) &
    (selected_data_midnight["station_type"] == "ML")
].copy()

mainline_midnight_data["avg_speed"] = pd.to_numeric(
    mainline_midnight_data["avg_speed"],
    errors="coerce"
)

median_speed_by_station = (
    mainline_midnight_data
    .groupby("station_id", as_index=False)["avg_speed"]
    .median()
)

# IMPORTANT:
# Keep this column name as "median_speed"
# because the next block uses median_speed_df["median_speed"]
median_speed_by_station.columns = ["station_id", "median_speed"]

# Optional clean display table, but do NOT change median_speed_by_station
median_speed_display = selected_metadata_clean[
    selected_metadata_clean["station_id"].isin(mainline_ids)
][
    ["station_id", "station_name", "absolute_postmile"]
].merge(
    median_speed_by_station,
    on="station_id",
    how="left"
).sort_values("absolute_postmile").reset_index(drop=True)

print("=== Median Free-Flow Speed by Mainline Station ===")
display(median_speed_display)

=== Median Free-Flow Speed by Mainline Station ===


,station_id,station_name,absolute_postmile,median_speed
0,1201419,RED HILL,8.17,64.20
1,1201469,BRISTOL 1,9.31,65.35
2,1201497,FAIRVIEW,10.05,69.85
3,1201525,HARBOR 1,10.97,68.40
4,1201558,HARBOR 2,11.27,68.70
5,1201589,EUCLID,12.27,68.05
6,1201620,TALBERT,13.07,67.95


In [178]:

# Verify median free-flow speed calculation
mainline_midnight_check = selected_data_midnight[
    (selected_data_midnight["station_id"].isin(mainline_ids)) &
    (selected_data_midnight["station_type"] == "ML")
].copy()

mainline_midnight_check["avg_speed"] = pd.to_numeric(
    mainline_midnight_check["avg_speed"],
    errors="coerce"
)

speed_check = (
    mainline_midnight_check
    .groupby("station_id")["avg_speed"]
    .agg(
        count="count",
        min_speed="min",
        median_speed="median",
        max_speed="max"
    )
    .reset_index()
)

speed_check = speed_check.merge(
    selected_metadata_clean[["station_id", "station_name"]],
    on="station_id",
    how="left"
)

speed_check = speed_check[
    ["station_id", "station_name", "count", "min_speed", "median_speed", "max_speed"]
]

display(speed_check)

,station_id,station_name,count,min_speed,median_speed,max_speed
0,1201419,RED HILL,48,47.3,64.20,78.7
1,1201469,BRISTOL 1,48,49.0,65.35,69.4
2,1201497,FAIRVIEW,48,39.7,69.85,74.4
3,1201525,HARBOR 1,48,63.4,68.40,72.9
4,1201558,HARBOR 2,48,64.9,68.70,73.8
5,1201589,EUCLID,48,63.4,68.05,74.3
6,1201620,TALBERT,48,62.5,67.95,73.9


In [179]:
# 6. Segment free-flow speed calculation
# Purpose:
# Use the median free-flow speed at each pair of neighboring
# mainline stations to estimate segment free-flow speed.

# segment free flow speed
def segment_free_flow_speed(upstream_id, downstream_id, median_speed_df):
    v_upstream = median_speed_df.loc[
        median_speed_df["station_id"] == upstream_id,
        "median_speed"
    ].iloc[0]

    v_downstream = median_speed_df.loc[
        median_speed_df["station_id"] == downstream_id,
        "median_speed"
    ].iloc[0]

    return (2 * v_upstream * v_downstream) / (v_upstream + v_downstream)

# free flow speed between each segment
v_ff_1 = segment_free_flow_speed(1201419, 1201469, median_speed_by_station)
v_ff_2 = segment_free_flow_speed(1201469, 1201497, median_speed_by_station)
v_ff_3 = segment_free_flow_speed(1201497, 1201525, median_speed_by_station)
v_ff_4 = segment_free_flow_speed(1201525, 1201558, median_speed_by_station)
v_ff_5 = segment_free_flow_speed(1201558, 1201589, median_speed_by_station)
v_ff_6 = segment_free_flow_speed(1201589, 1201620, median_speed_by_station)

segment_free_flow_df = pd.DataFrame({
    "segment": ["S1", "S2", "S3", "S4", "S5", "S6"],
    "from_station": ["RED HILL", "BRISTOL 1", "FAIRVIEW", "HARBOR 1", "HARBOR 2", "EUCLID"],
    "to_station": ["BRISTOL 1", "FAIRVIEW", "HARBOR 1", "HARBOR 2", "EUCLID", "TALBERT"],
    "v_ff_mph": [v_ff_1, v_ff_2, v_ff_3, v_ff_4, v_ff_5, v_ff_6]
})
segment_free_flow_df["v_ff_mph"] = segment_free_flow_df["v_ff_mph"].round(3)
display(segment_free_flow_df)

,segment,from_station,to_station,v_ff_mph
0,S1,RED HILL,BRISTOL 1,64.770
1,S2,BRISTOL 1,FAIRVIEW,67.525
2,S3,FAIRVIEW,HARBOR 1,69.117
3,S4,HARBOR 1,HARBOR 2,68.550
4,S5,HARBOR 2,EUCLID,68.373
5,S6,EUCLID,TALBERT,68.000


In [180]:
# 7. Free-flow travel time calculation
# Purpose:
# Convert each segment's free-flow speed into free-flow travel time.

# Formula:
# travel_time_hours = segment_length_miles / free_flow_speed_mph
# travel_time_minutes = travel_time_hours * 60
# segment length in miles

L_1 = 1.14
L_2 = 0.74
L_3 = 0.92
L_4 = 0.30
L_5 = 1.00
L_6 = 0.80

def segment_free_flow_travel_time(length, segment_free_flow_speed):
    travel_time_hour = length / segment_free_flow_speed
    travel_time_min = travel_time_hour * 60
    return travel_time_hour, travel_time_min

TT_ff_1_hr, TT_ff_1_min = segment_free_flow_travel_time(L_1, v_ff_1)
TT_ff_2_hr, TT_ff_2_min = segment_free_flow_travel_time(L_2, v_ff_2)
TT_ff_3_hr, TT_ff_3_min = segment_free_flow_travel_time(L_3, v_ff_3)
TT_ff_4_hr, TT_ff_4_min = segment_free_flow_travel_time(L_4, v_ff_4)
TT_ff_5_hr, TT_ff_5_min = segment_free_flow_travel_time(L_5, v_ff_5)
TT_ff_6_hr, TT_ff_6_min = segment_free_flow_travel_time(L_6, v_ff_6)

total_h = TT_ff_1_hr + TT_ff_2_hr + TT_ff_3_hr + TT_ff_4_hr + TT_ff_5_hr + TT_ff_6_hr
total_min = TT_ff_1_min + TT_ff_2_min + TT_ff_3_min + TT_ff_4_min + TT_ff_5_min + TT_ff_6_min

free_flow_tt_df = pd.DataFrame({
    "segment": [
        "Segment 1: RED HILL → BRISTOL 1",
        "Segment 2: BRISTOL 1 → FAIRVIEW",
        "Segment 3: FAIRVIEW → HARBOR 1",
        "Segment 4: HARBOR 1 → HARBOR 2",
        "Segment 5: HARBOR 2 → EUCLID",
        "Segment 6: EUCLID → TALBERT",
    ],
    "length_miles": [L_1, L_2, L_3, L_4, L_5, L_6],
    "free_flow_speed_mph": [v_ff_1, v_ff_2, v_ff_3, v_ff_4, v_ff_5, v_ff_6],
    "TT_ff_hr": [TT_ff_1_hr, TT_ff_2_hr, TT_ff_3_hr, TT_ff_4_hr, TT_ff_5_hr, TT_ff_6_hr],
    "TT_ff_min": [TT_ff_1_min, TT_ff_2_min, TT_ff_3_min, TT_ff_4_min, TT_ff_5_min, TT_ff_6_min],
})

print("Segment Free-Flow Travel Time ")
display(free_flow_tt_df)

print("Total free-flow travel time:")
print("Hours:", round(total_h, 3))
print("Minutes:", round(total_min, 3))

Segment Free-Flow Travel Time 


,segment,length_miles,free_flow_speed_mph,TT_ff_hr,TT_ff_min
0,Segment 1: RED HILL → BRISTOL 1,1.14,64.769896,0.017601,1.056046
1,Segment 2: BRISTOL 1 → FAIRVIEW,0.74,67.525111,0.010959,0.657533
2,Segment 3: FAIRVIEW → HARBOR 1,0.92,69.117396,0.013311,0.798641
3,Segment 4: HARBOR 1 → HARBOR 2,0.30,68.549672,0.004376,0.262583
4,Segment 5: HARBOR 2 → EUCLID,1.00,68.373455,0.014626,0.877534
5,Segment 6: EUCLID → TALBERT,0.80,67.999963,0.011765,0.705883


Total free-flow travel time:
Hours: 0.073
Minutes: 4.358


In [204]:
# segment travel time using observed segment speed and segment length
def segment_observed_travel_time(length, observed_segment_speed):
    travel_time_hour = length / observed_segment_speed
    travel_time_min = travel_time_hour * 60
    return travel_time_hour, travel_time_min

TT_obs_1_hr, TT_obs_1_min = segment_observed_travel_time(L_1, v_obs_1)
TT_obs_2_hr, TT_obs_2_min = segment_observed_travel_time(L_2, v_obs_2)
TT_obs_3_hr, TT_obs_3_min = segment_observed_travel_time(L_3, v_obs_3)
TT_obs_4_hr, TT_obs_4_min = segment_observed_travel_time(L_4, v_obs_4)
TT_obs_5_hr, TT_obs_5_min = segment_observed_travel_time(L_5, v_obs_5)
TT_obs_6_hr, TT_obs_6_min = segment_observed_travel_time(L_6, v_obs_6)

print(round(TT_obs_1_hr, 3), round(TT_obs_1_min, 3))
print(round(TT_obs_2_hr, 3), round(TT_obs_2_min, 3))
print(round(TT_obs_3_hr, 3), round(TT_obs_3_min, 3))
print(round(TT_obs_4_hr, 3), round(TT_obs_4_min, 3))
print(round(TT_obs_5_hr, 3), round(TT_obs_5_min, 3))
print(round(TT_obs_6_hr, 3), round(TT_obs_6_min, 3))

0.029 1.723
0.015 0.874
0.022 1.299
0.01 0.619
0.04 2.428
0.03 1.788


In [182]:
## calculate Segment delay(this is for each car), which is equal to the difference between observed travel time and  segment free flow travel time

def segment_delay(segment_observed_travel_time,segment_free_flow_travel_time):
    segment_delay = segment_observed_travel_time - segment_free_flow_travel_time
    return segment_delay


In [183]:
delay_1 = segment_delay(TT_obs_1_hr, TT_ff_1_hr)
delay_2 = segment_delay(TT_obs_2_hr, TT_ff_2_hr)
delay_3 = segment_delay(TT_obs_3_hr, TT_ff_3_hr)
delay_4 = segment_delay(TT_obs_4_hr, TT_ff_4_hr)
delay_5 = segment_delay(TT_obs_5_hr, TT_ff_5_hr)
delay_6 = segment_delay(TT_obs_6_hr, TT_ff_6_hr)

delay_1_min = segment_delay(TT_obs_1_min, TT_ff_1_min)
delay_2_min = segment_delay(TT_obs_2_min, TT_ff_2_min)
delay_3_min = segment_delay(TT_obs_3_min, TT_ff_3_min)
delay_4_min = segment_delay(TT_obs_4_min, TT_ff_4_min)
delay_5_min = segment_delay(TT_obs_5_min, TT_ff_5_min)
delay_6_min = segment_delay(TT_obs_6_min, TT_ff_6_min)

print(round(delay_1, 3))
print(round(delay_2, 3))
print(round(delay_3, 3))
print(round(delay_4, 3))
print(round(delay_5, 3))
print(round(delay_6, 3))

print(round(delay_1_min, 3))
print(round(delay_2_min, 3))
print(round(delay_3_min, 3))
print(round(delay_4_min, 3))
print(round(delay_5_min, 3))
print(round(delay_6_min, 3))

total_delay_hours = delay_1 + delay_2 + delay_3 + delay_4 + delay_5 + delay_6
print(round(total_delay_hours, 3))

total_delay_min = delay_1_min + delay_2_min + delay_3_min + delay_4_min + delay_5_min + delay_6_min
print(round(total_delay_min, 3))



0.011
0.004
0.008
0.006
0.026
0.018
0.667
0.217
0.5
0.356
1.551
1.083
0.073
4.374


In [184]:
# Total mainline delay for each segment
# delay_i is in vehicle-hours per vehicle
# flow_i is vehicles during the 5-minute interval
# result is vehicle-hours during the 5-minute interval
def total_mainline_delay(segment_delay, total_flow):
    total_mainline_delay = segment_delay * total_flow
    return total_mainline_delay


In [185]:
Q_out_1= 681
Q_out_2 = 622
Q_out_3 = 737
Q_out_4 = 599
Q_out_5 = 680
Q_out_6 = 550

mainline_delay_1_min = delay_1_min * Q_out_1
mainline_delay_2_min = delay_2_min * Q_out_2
mainline_delay_3_min = delay_3_min * Q_out_3
mainline_delay_4_min = delay_4_min * Q_out_4
mainline_delay_5_min = delay_5_min * Q_out_5
mainline_delay_6_min = delay_6_min* Q_out_6

mainline_delay_1_hr = mainline_delay_1_min / 60
mainline_delay_2_hr = mainline_delay_2_min / 60
mainline_delay_3_hr = mainline_delay_3_min / 60
mainline_delay_4_hr = mainline_delay_4_min / 60
mainline_delay_5_hr = mainline_delay_5_min / 60
mainline_delay_6_hr = mainline_delay_6_min / 60

print(round(mainline_delay_1_hr, 3))
print(round(mainline_delay_2_hr, 3))
print(round(mainline_delay_3_hr, 3))
print(round(mainline_delay_4_hr, 3))
print(round(mainline_delay_5_hr, 3))
print(round(mainline_delay_6_hr, 3))

print(round(mainline_delay_1_min, 3))
print(round(mainline_delay_2_min, 3))
print(round(mainline_delay_3_min, 3))
print(round(mainline_delay_4_min, 3))
print(round(mainline_delay_5_min, 3))
print(round(mainline_delay_6_min, 3))

total_flow_based_delay_min = sum([mainline_delay_1_min,mainline_delay_2_min,mainline_delay_3_min,mainline_delay_4_min,mainline_delay_5_min, mainline_delay_6_min])

print('Total flow based delay in min:', total_flow_based_delay_min)

7.57
2.248
6.147
3.559
17.576
9.923
454.209
134.906
368.826
213.51
1054.537
595.384
Total flow based delay in min: 2821.3730867073427


# we need to calculate state based total delay

---------------------------------------------------------------
6-cell, 5-min prototype state-based delay (Section 8.2 of note)
NOTE: superseded by the 8-cell, 30-sec model in Part 3.
X_final here uses observed Q_out (no CTM sending/receiving),
which is why this number CANNOT be compared to the
26532 vs 15047 veh-min totals from the 8-cell ADMM model.
Kept only as a sanity check of the state-based formula.


# This computes the state-based baseline that matches note:
$$D_M,i(t) = TTT_i(t) - (Q_out,i + f_i) * TT_ff,i$$

where:
$$TTT_i(t) = ((X_i,t-1 + X_i,t) / 2) * DeltaT$$

# Units:
X = vehicles

DeltaT = minutes

TTT = vehicle-minutes

TT_ff = minutes

$(Q_out + f_i) * TT_ff$ = vehicle-minutes

$D_M,i = vehicle-minutes$



In [186]:

# 1. these values are previesly calculated

delta_t_min = 5.0
delta_t_hr = delta_t_min / 60.0
delta_t_sec = delta_t_min * 60.0


X_initial = {
    "Segment 1": 228.426,
    "Segment 2": 115.327,
    "Segment 3": 182.351,
    "Segment 4": 82.277,
    "Segment 5": 312.707,
    "Segment 6": 224.706
}

X_final = {
    "Segment 1": 280.426,
    "Segment 2": 174.327,
    "Segment 3": 179.351,
    "Segment 4": 194.277,
    "Segment 5": 287.707,
    "Segment 6": 354.706
}

Q_out = {
    "Segment 1": 681,
    "Segment 2": 622,
    "Segment 3": 737,
    "Segment 4": 599,
    "Segment 5": 680,
    "Segment 6": 550
}

f_sum = {
    # BRISTOL off-ramp missing -> assumed 0,
    # we used different values such as [10,20,50,80] to see its impact on the delay
    "Segment 1": 0,
    "Segment 2": 0,
    "Segment 3": 0,
    "Segment 4": 82,
    "Segment 5": 8,
    "Segment 6": 0
}

#TT_ff_1_min is calculated above
TT_ff_min = {
    "Segment 1": TT_ff_1_min,
    "Segment 2": TT_ff_2_min,
    "Segment 3": TT_ff_3_min,
    "Segment 4": TT_ff_4_min,
    "Segment 5": TT_ff_5_min,
    "Segment 6": TT_ff_6_min
}



# 2. State-based mainline delay function
def state_based_mainline_delay(x_prev, x_curr, q_out, f_out, tt_ff_min, delta_t_min):
    x_avg = (x_prev + x_curr) / 2.0
    ttt = x_avg * delta_t_min
    free_flow_component = (q_out + f_out) * tt_ff_min
    delay = ttt - free_flow_component
    return x_avg, ttt, free_flow_component, delay


# 3. Compute state-based delay for each segment
results = []
total_ttt = 0.0
total_ff_component = 0.0
total_state_delay = 0.0

for segment in X_initial:
    x_avg, ttt, ff_component, delay = state_based_mainline_delay(
        X_initial[segment],
        X_final[segment],
        Q_out[segment],
        f_sum[segment],
        TT_ff_min[segment],
        delta_t_min
    )

    total_ttt += ttt
    total_ff_component += ff_component
    total_state_delay += delay

    results.append({
        "Segment": segment,
        "X_prev": round(X_initial[segment], 3),
        "X_curr": round(X_final[segment], 3),
        "X_avg": round(x_avg, 3),
        "TTT (veh-min)": round(ttt, 3),
        "Q_out": Q_out[segment],
        "f_out": f_sum[segment],
        "TT_ff (min)": round(TT_ff_min[segment], 3),
        "(Q_out + f_out) * TT_ff (veh-min)": round(ff_component, 3),
        "State-based delay (veh-min)": round(delay, 3)
    })



# 4. Print clean table
state_delay_df = pd.DataFrame(results)
print(state_delay_df.to_string(index=False))

print("\nTOTAL TTT (veh-min):", round(total_ttt, 3))
print("TOTAL free-flow component (veh-min):", round(total_ff_component, 3))
print("TOTAL state-based mainline delay baseline (veh-min):", round(total_state_delay, 3))

  Segment  X_prev  X_curr   X_avg  TTT (veh-min)  Q_out  f_out  TT_ff (min)  (Q_out + f_out) * TT_ff (veh-min)  State-based delay (veh-min)
Segment 1 228.426 280.426 254.426       1272.130    681      0        1.056                            719.167                      552.963
Segment 2 115.327 174.327 144.827        724.135    622      0        0.658                            408.986                      315.149
Segment 3 182.351 179.351 180.851        904.255    737      0        0.799                            588.599                      315.656
Segment 4  82.277 194.277 138.277        691.385    599     82        0.263                            178.819                      512.566
Segment 5 312.707 287.707 300.207       1501.035    680      8        0.878                            603.743                      897.292
Segment 6 224.706 354.706 289.706       1448.530    550      0        0.706                            388.236                     1060.294

TOTAL TTT (veh-min)

## Local Road Delay Calculation
# below calculation came from using the Station 5 min dataset for the district 12
# we use the result as starting point to see what actually happened on the local road with "no-control" baseline


In [187]:
#selected dataset 2026-01-08 08:00:00 -2026-01-08 08:00:00
# we only kept the On-ramp ID

ids_to_keep = [
    1201419, 1201469, 1201497, 1201525, 1201558, 1201589, 1201620,
    1201460, 1201490, 1201517, 1201548, 1201580,
    1201465, 1201554, 1201585
]

start_time = "2026-01-08 08:00:00"
end_time = "2026-01-08 08:00:00"

filtered_data = station_data[
    (station_data[1].isin(ids_to_keep)) &
    (station_data[0] >= start_time) &
    (station_data[0] <= end_time)
]

selected_data_morning = filtered_data[[0, 1, 5, 9, 10, 11]]
selected_data_morning.columns = ["timestamp","station_id","station_type","total_flow","avg_occupancy","avg_speed"]


pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

print(selected_data_morning.to_string())


                 timestamp  station_id station_type  total_flow  avg_occupancy  avg_speed
248441 2026-01-08 08:00:00     1201419           ML       645.0         0.1582       39.4
248447 2026-01-08 08:00:00     1201460           OR        88.0         0.0840        NaN
248448 2026-01-08 08:00:00     1201465           FR         NaN            NaN        NaN
248450 2026-01-08 08:00:00     1201469           ML       681.0         0.1384       40.0
248454 2026-01-08 08:00:00     1201490           OR        39.0         0.8700        NaN
248456 2026-01-08 08:00:00     1201497           ML       622.0         0.0525       69.5
248460 2026-01-08 08:00:00     1201517           OR        73.0         0.0980        NaN
248462 2026-01-08 08:00:00     1201525           ML       737.0         0.1677       30.6
248466 2026-01-08 08:00:00     1201548           OR        56.0         0.0980        NaN
248467 2026-01-08 08:00:00     1201554           FR        82.0         0.0283        NaN
248469 202

In [188]:
# ramp queue formula and local road delay formulas

def ramp_queue(R_prev, a_t, u_t):
    R_t = R_prev + a_t - u_t
    return R_t

def local_road_delay(R_prev, R_t, delta_t):
    delay = ((R_prev + R_t) / 2) * delta_t
    return delay

# 08:00 on-ramp flows
u = {
    "u1": 88,
    "u2": 39,
    "u3": 73,
    "u4": 56,
    "u5": 64
}

# queue benchmark cases
R_prev_cases = [0, 10, 20, 30]

delta_t = 5


# arrival assumptions(a_t)
assumptions = {
    "Assumption 2: a = u": 1.0,
    "Assumption 3: a = 1.5u": 1.5,
    "Assumption 4: a = 2u": 2.0
}

for assumption_name, multiplier in assumptions.items():
    print("\n" + assumption_name)

    for R_prev in R_prev_cases:
        print("\nR_prev =", R_prev)

        total_delay=0



        for ramp_name, u_t in u.items():
            a_t = multiplier * u_t
            R_t = ramp_queue(R_prev, a_t, u_t)
            D_t = local_road_delay(R_prev, R_t, delta_t)

            total_delay += D_t

            print(
                ramp_name,
                "u =", u_t,
                "a =", a_t,
                "R_t =", R_t,
                "D_L =", D_t
            )
        print("Total local delay =", total_delay)


Assumption 2: a = u

R_prev = 0
u1 u = 88 a = 88.0 R_t = 0.0 D_L = 0.0
u2 u = 39 a = 39.0 R_t = 0.0 D_L = 0.0
u3 u = 73 a = 73.0 R_t = 0.0 D_L = 0.0
u4 u = 56 a = 56.0 R_t = 0.0 D_L = 0.0
u5 u = 64 a = 64.0 R_t = 0.0 D_L = 0.0
Total local delay = 0.0

R_prev = 10
u1 u = 88 a = 88.0 R_t = 10.0 D_L = 50.0
u2 u = 39 a = 39.0 R_t = 10.0 D_L = 50.0
u3 u = 73 a = 73.0 R_t = 10.0 D_L = 50.0
u4 u = 56 a = 56.0 R_t = 10.0 D_L = 50.0
u5 u = 64 a = 64.0 R_t = 10.0 D_L = 50.0
Total local delay = 250.0

R_prev = 20
u1 u = 88 a = 88.0 R_t = 20.0 D_L = 100.0
u2 u = 39 a = 39.0 R_t = 20.0 D_L = 100.0
u3 u = 73 a = 73.0 R_t = 20.0 D_L = 100.0
u4 u = 56 a = 56.0 R_t = 20.0 D_L = 100.0
u5 u = 64 a = 64.0 R_t = 20.0 D_L = 100.0
Total local delay = 500.0

R_prev = 30
u1 u = 88 a = 88.0 R_t = 30.0 D_L = 150.0
u2 u = 39 a = 39.0 R_t = 30.0 D_L = 150.0
u3 u = 73 a = 73.0 R_t = 30.0 D_L = 150.0
u4 u = 56 a = 56.0 R_t = 30.0 D_L = 150.0
u5 u = 64 a = 64.0 R_t = 30.0 D_L = 150.0
Total local delay = 750.0

Assum

## Fairness penalty calculation
## below , we calculated the fairness penalty using maximum ramp capacity, current queue on the ramp




In [189]:
## maximum ramp capacity calculation

ave_veh_length= 25

ramp_length={
    "Bristol 1": 716.73,
    "Fairview": 1808.89,
    "Harbor 1": 1404.20,
    "Harbor 2":1811.02,
    "Euclid": 610.24}

number_of_lanes={"Bristol 1":1,
    "Fairview": 1,
    "Harbor 1": 1,
    "Harbor 2":1,
    "Euclid": 2}

def R_max (ave_veh_length, ramp_length, number_of_lanes):
    R_max={}

    for ramps in ramp_length:
        R_max[ramps] =(ramp_length[ramps] * number_of_lanes[ramps])/ave_veh_length

    return R_max

R_max_value=R_max(ave_veh_length, ramp_length, number_of_lanes)
for ramps in R_max_value:
    print(ramps,"max ramp capacity:",  round(R_max_value[ramps], 0))



Bristol 1 max ramp capacity: 29.0
Fairview max ramp capacity: 72.0
Harbor 1 max ramp capacity: 56.0
Harbor 2 max ramp capacity: 72.0
Euclid max ramp capacity: 49.0


In [190]:
## stress index calculation
## result should be between 0-1 , 1 being the ramp is at 100% capacity
## if the result exceeds 1, the ramp is spilling back cars into local road

#selected ramps total flow
u = {
    "Bristol 1": 88,
    "Fairview": 39,
    "Harbor 1": 73,
    "Harbor 2": 56,
    "Euclid": 64
}

R_max = {
    "Bristol 1": 29.0,
    "Fairview": 72.0,
    "Harbor 1": 56.0,
    "Harbor 2": 72.0,
    "Euclid": 49.0
}

def ramp_queue(R_prev, a_t, u_t):
    return R_prev + a_t - u_t



def stress_index(R_t, R_max):
    return R_t/R_max

for assumption_name, multiplier in assumptions.items():
    print("\n" + assumption_name)
    for R_prev in R_prev_cases:
        print("\nR_prev =", R_prev)
        stress_all = {}
        for ramp, u_t in u.items():
            a_t = multiplier * u_t
            R_t = ramp_queue(R_prev, a_t, u_t)
            phi_t = stress_index(R_t, R_max[ramp])

            stress_all[ramp] = phi_t

            print(ramp,":", "R_t =", round(R_t, 3), "stress =", round(phi_t, 3))







Assumption 2: a = u

R_prev = 0
Bristol 1 : R_t = 0.0 stress = 0.0
Fairview : R_t = 0.0 stress = 0.0
Harbor 1 : R_t = 0.0 stress = 0.0
Harbor 2 : R_t = 0.0 stress = 0.0
Euclid : R_t = 0.0 stress = 0.0

R_prev = 10
Bristol 1 : R_t = 10.0 stress = 0.345
Fairview : R_t = 10.0 stress = 0.139
Harbor 1 : R_t = 10.0 stress = 0.179
Harbor 2 : R_t = 10.0 stress = 0.139
Euclid : R_t = 10.0 stress = 0.204

R_prev = 20
Bristol 1 : R_t = 20.0 stress = 0.69
Fairview : R_t = 20.0 stress = 0.278
Harbor 1 : R_t = 20.0 stress = 0.357
Harbor 2 : R_t = 20.0 stress = 0.278
Euclid : R_t = 20.0 stress = 0.408

R_prev = 30
Bristol 1 : R_t = 30.0 stress = 1.034
Fairview : R_t = 30.0 stress = 0.417
Harbor 1 : R_t = 30.0 stress = 0.536
Harbor 2 : R_t = 30.0 stress = 0.417
Euclid : R_t = 30.0 stress = 0.612

Assumption 3: a = 1.5u

R_prev = 0
Bristol 1 : R_t = 44.0 stress = 1.517
Fairview : R_t = 19.5 stress = 0.271
Harbor 1 : R_t = 36.5 stress = 0.652
Harbor 2 : R_t = 28.0 stress = 0.389
Euclid : R_t = 32.0 str

## Above Result is based on different scenrios set up for Previews Queue , newly arrived cars .
## for previews queue , we used numbers {0,10,20,30}
## for arrival car , we used : arrival = {discharged, 1.5* discharged , 2.0* discharged}

## Some results were above 1, in some extreme cases , 3 and 4 . we will treat anything above 1 as spillback


In [191]:
## fariness penalty

## penalty multiplier
gamma_values = [0, 0.5, 0.8, 1, 5]

#discharged cars
u = {
    "Bristol 1": 88,
    "Fairview": 39,
    "Harbor 1": 73,
    "Harbor 2": 56,
    "Euclid": 64
}


R_prev_cases = [0, 10, 20, 30]

assumptions = {
    "Assumption 2: a = u": 1.0,
    "Assumption 3: a = 1.5u": 1.5,
    "Assumption 4: a = 2u": 2.0
}


R_max_value = {
    "Bristol 1": 29.0,
    "Fairview": 72.0,
    "Harbor 1": 56.0,
    "Harbor 2": 72.0,
    "Euclid": 49.0
}


def ramp_queue(R_prev, a_t, u_t):
    return R_prev + a_t - u_t

#making sure that code ignores strees index if its > 1
def capped_stress_index(R_t, R_max):
    raw_stress = R_t / R_max
    capped_stress = min(raw_stress, 1)
    return capped_stress

# fairness penalty calculation
def fairness_penalty(gamma, stress_dict):
    ramps = list(stress_dict.keys())
    penalty = 0

    for i in range(len(ramps)):
        for j in range(i + 1, len(ramps)):
            phi_i = stress_dict[ramps[i]]
            phi_j = stress_dict[ramps[j]]
            penalty += (phi_i - phi_j) ** 2

    return gamma * penalty

for assumption_name, multiplier in assumptions.items():
    print("\n" + assumption_name)

    for R_prev in R_prev_cases:
        print("\nR_prev =", R_prev)

        stress_all = {}

        for ramp, u_t in u.items():
            a_t = multiplier * u_t
            R_t = ramp_queue(R_prev, a_t, u_t)
            phi_t = capped_stress_index(R_t, R_max_value[ramp])

            stress_all[ramp] = phi_t

            print(ramp, "R_t =", round(R_t, 3), "capped stress =", round(phi_t, 3)
                                                                                   )

        for gamma in gamma_values:
            L_fair = fairness_penalty(gamma, stress_all)
            print("gamma =", gamma, "," " fairness penalty =", round(L_fair, 3))


Assumption 2: a = u

R_prev = 0
Bristol 1 R_t = 0.0 capped stress = 0.0
Fairview R_t = 0.0 capped stress = 0.0
Harbor 1 R_t = 0.0 capped stress = 0.0
Harbor 2 R_t = 0.0 capped stress = 0.0
Euclid R_t = 0.0 capped stress = 0.0
gamma = 0 , fairness penalty = 0.0
gamma = 0.5 , fairness penalty = 0.0
gamma = 0.8 , fairness penalty = 0.0
gamma = 1 , fairness penalty = 0.0
gamma = 5 , fairness penalty = 0.0

R_prev = 10
Bristol 1 R_t = 10.0 capped stress = 0.345
Fairview R_t = 10.0 capped stress = 0.139
Harbor 1 R_t = 10.0 capped stress = 0.179
Harbor 2 R_t = 10.0 capped stress = 0.139
Euclid R_t = 10.0 capped stress = 0.204
gamma = 0 , fairness penalty = 0.0
gamma = 0.5 , fairness penalty = 0.072
gamma = 0.8 , fairness penalty = 0.116
gamma = 1 , fairness penalty = 0.145
gamma = 5 , fairness penalty = 0.723

R_prev = 20
Bristol 1 R_t = 20.0 capped stress = 0.69
Fairview R_t = 20.0 capped stress = 0.278
Harbor 1 R_t = 20.0 capped stress = 0.357
Harbor 2 R_t = 20.0 capped stress = 0.278
Eucl

# Capacity Penalty

## below calculation is for mainline capacity calculation and penalty.
## we split the capacity into three terms : doorway capacity, Safe occupancy threshold, and  physical capacity




## Below calculation are for doorway capacity


In [192]:
# Doorway capacity calculation
# known capacities are already in vehicles per 5-minute interval

doorway_capacity_given = {
    "Bristol 1": 895,
    "Harbor 1": 1029,
    "Harbor 2": 860,
    "Euclid": 925,
    "Talbert": 833
}

number_of_lanes = {
    "RED HILL": 5,
    "Bristol 1": 5,
    "FAIRVIEW": 5,
    "Harbor 1": 6,
    "Harbor 2": 5,
    "Euclid": 5,
    "Talbert": 5
}

# find per-lane doorway capacity for stations with known capacity
def per_lane_doorway_cap(doorway_capacity_dict, lane_dict):
    per_lane = {}

    for station in doorway_capacity_dict:
        per_lane[station] = doorway_capacity_dict[station] / lane_dict[station]

    return per_lane

# average per-lane doorway capacity from known stations
def average_per_lane_capacity(per_lane_dict):
    total = 0

    for station in per_lane_dict:
        total += per_lane_dict[station]

    return total / len(per_lane_dict)

# estimate doorway capacity for stations with missing values
def doorway_capacity_not_given(avg_per_lane_cap, lane_dict, missing_stations):
    estimated = {}

    for station in missing_stations:
        estimated[station] = avg_per_lane_cap * lane_dict[station]

    return estimated


per_lane_capacity = per_lane_doorway_cap(doorway_capacity_given, number_of_lanes)

print("Per-lane doorway capacity for known stations:")
for station in per_lane_capacity:
    print(station, round(per_lane_capacity[station], 0))

avg_per_lane_cap = round(average_per_lane_capacity(per_lane_capacity))
print("\nAverage per-lane doorway capacity:", avg_per_lane_cap)

missing_stations = ["RED HILL", "FAIRVIEW"]
estimated_capacity = doorway_capacity_not_given(avg_per_lane_cap, number_of_lanes, missing_stations)

print("\nEstimated doorway capacity for missing stations:")
for station in estimated_capacity:
    print(station, round(estimated_capacity[station], 0))

Per-lane doorway capacity for known stations:
Bristol 1 179.0
Harbor 1 172.0
Harbor 2 172.0
Euclid 185.0
Talbert 167.0

Average per-lane doorway capacity: 175

Estimated doorway capacity for missing stations:
RED HILL 875
FAIRVIEW 875


# Final Doorway Capacity


In [193]:
final_doorway_capacity={
    "RED HILL":875,
    "Bristol 1": 895,
    "FAIRVIEW": 875,
    "Harbor 1": 1029,
    "Harbor 2": 860,
    "Euclid": 925,
    "Talbert": 833

}

## Physical capcity calculation

In [194]:
# Physical capacity of each segment

jam_density = 193  # veh/mi/ln

segment_length = {
    "Segment 1": 1.14,
    "Segment 2": 0.74,
    "Segment 3": 0.92,
    "Segment 4": 0.30,
    "Segment 5": 1.00,
    "Segment 6": 0.80
}

number_of_lanes = {
    "Segment 1": 5,
    "Segment 2": 5,
    "Segment 3": 6,
    "Segment 4": 5,
    "Segment 5": 5,
    "Segment 6": 5
}

def physical_capacity(jam_density, segment_length, number_of_lanes):
    N_max = {}
    for segment in segment_length:
        N_max[segment] = jam_density * segment_length[segment] * number_of_lanes[segment]
    return N_max
N_max_value = physical_capacity(jam_density, segment_length, number_of_lanes)
for segment in N_max_value:
    print(segment, "physical capacity =", round(N_max_value[segment], 0))

Segment 1 physical capacity = 1100.0
Segment 2 physical capacity = 714.0
Segment 3 physical capacity = 1065.0
Segment 4 physical capacity = 290.0
Segment 5 physical capacity = 965.0
Segment 6 physical capacity = 772.0


## Safe occupancy threshold calculation

In [195]:
## safe occupancy threshold calculation

threshold_multipliers = [0.3, 0.5, 0.7]

def safe_occupancy_thresholds(threshold_multipliers, N_max_value):
    safe_occupancy = {}

    for multiplier in threshold_multipliers:
        label = f"eta = {multiplier}"
        safe_occupancy[label] = {}

        for segment in N_max_value:
            safe_occupancy[label][segment] = multiplier * N_max_value[segment]

    return safe_occupancy

safe_occupancy = safe_occupancy_thresholds(threshold_multipliers, N_max_value)

for label in safe_occupancy:
    print(label)
    for segment in safe_occupancy[label]:
        print(segment, "X_safe =", round(safe_occupancy[label][segment], 3))
    print()

eta = 0.3
Segment 1 X_safe = 330.03
Segment 2 X_safe = 214.23
Segment 3 X_safe = 319.608
Segment 4 X_safe = 86.85
Segment 5 X_safe = 289.5
Segment 6 X_safe = 231.6

eta = 0.5
Segment 1 X_safe = 550.05
Segment 2 X_safe = 357.05
Segment 3 X_safe = 532.68
Segment 4 X_safe = 144.75
Segment 5 X_safe = 482.5
Segment 6 X_safe = 386.0

eta = 0.7
Segment 1 X_safe = 770.07
Segment 2 X_safe = 499.87
Segment 3 X_safe = 745.752
Segment 4 X_safe = 202.65
Segment 5 X_safe = 675.5
Segment 6 X_safe = 540.4



## Below calculation are for doorway capacity


## Doorway Capacity penalty calculation
## Note , we used syntatic data for flowrate (Q_in) as well as ramp flow (U_sum) to varify if the calculation works

In [196]:

# doorway capacity penalty
lambda_1_values = [0, 0.5, 1, 5]
#actual mainline flow
Q_in = {
    "Segment 1": 645,
    "Segment 2": 681,
    "Segment 3": 622,
    "Segment 4": 737,
    "Segment 5": 599,
    "Segment 6": 680
}

#synthetic mainline flow for comfirming penalty calculation is working
# we added 300 to each mainline flow
Q_in_fake = {
    "Segment 1": 945,
    "Segment 2": 981,
    "Segment 3": 922,
    "Segment 4": 1037,
    "Segment 5": 899,
    "Segment 6": 980
}
#ramp flow
u_sum = {
    "Segment 1": 88,
    "Segment 2": 0,
    "Segment 3": 39 + 73,
    "Segment 4": 56,
    "Segment 5": 64,
    "Segment 6": 0
}
## synthetic ramp flow for comfirming penalty calculation is working
## we added 20 for each
u_sum_fake= {
    "Segment 1": 108,
    "Segment 2": 0,
    "Segment 3": 59 + 93,
    "Segment 4": 76,
    "Segment 5": 84,
    "Segment 6": 0
}

#calculated doorway capacity
C_i = {
    "Segment 1": 895,
    "Segment 2": 875,
    "Segment 3": 1029,
    "Segment 4": 860,
    "Segment 5": 925,
    "Segment 6": 833
}

def doorway_capacity_penalty(lambda_1, q_in, u_total, capacity):
    overflow = max(0, q_in + u_total - capacity)
    penalty = lambda_1 * (overflow ** 2)
    return overflow, penalty

for lambda_1 in lambda_1_values:
    print("\nlambda_1 =", lambda_1)
    total_penalty = 0

    for segment in Q_in:
        overflow, penalty = doorway_capacity_penalty(
            lambda_1,
            Q_in_fake[segment],
            u_sum_fake[segment],
            C_i[segment]
        )

        total_penalty += penalty

        print(segment,
              "Q_in + u =", Q_in_fake[segment] + u_sum_fake[segment],
              ",","C_i =", C_i[segment],
              ",","overflow =", overflow,
              "penalty =", penalty)

    print("Total doorway penalty =", total_penalty)


lambda_1 = 0
Segment 1 Q_in + u = 1053 , C_i = 895 , overflow = 158 penalty = 0
Segment 2 Q_in + u = 981 , C_i = 875 , overflow = 106 penalty = 0
Segment 3 Q_in + u = 1074 , C_i = 1029 , overflow = 45 penalty = 0
Segment 4 Q_in + u = 1113 , C_i = 860 , overflow = 253 penalty = 0
Segment 5 Q_in + u = 983 , C_i = 925 , overflow = 58 penalty = 0
Segment 6 Q_in + u = 980 , C_i = 833 , overflow = 147 penalty = 0
Total doorway penalty = 0

lambda_1 = 0.5
Segment 1 Q_in + u = 1053 , C_i = 895 , overflow = 158 penalty = 12482.0
Segment 2 Q_in + u = 981 , C_i = 875 , overflow = 106 penalty = 5618.0
Segment 3 Q_in + u = 1074 , C_i = 1029 , overflow = 45 penalty = 1012.5
Segment 4 Q_in + u = 1113 , C_i = 860 , overflow = 253 penalty = 32004.5
Segment 5 Q_in + u = 983 , C_i = 925 , overflow = 58 penalty = 1682.0
Segment 6 Q_in + u = 980 , C_i = 833 , overflow = 147 penalty = 10804.5
Total doorway penalty = 63603.5

lambda_1 = 1
Segment 1 Q_in + u = 1053 , C_i = 895 , overflow = 158 penalty = 2496

## Doorway capacity penalty calculation

## Note: we used synatic data for final state of the segment (X_final) to vafity the calculation works or not

In [197]:
## Physical Capacity Penalty Calculation

lambda_3_values = [0, 0.5, 1, 5]

# initial vehicles already inside each segment at start of interval

#segment length
segment_lengths = {
    "Segment 1": 1.14,
    "Segment 2": 0.74,
    "Segment 3": 0.92,
    "Segment 4": 0.30,
    "Segment 5": 1.00,
    "Segment 6": 0.80
}


#segments between 2 stations
segment_stations = {
    "Segment 1": (1201419, 1201469),
    "Segment 2": (1201469, 1201497),
    "Segment 3": (1201497, 1201525),
    "Segment 4": (1201525, 1201558),
    "Segment 5": (1201558, 1201589),
    "Segment 6": (1201589, 1201620)
}


#mainline for starting of each segment
mainline_station_ids = [1201419, 1201469, 1201497, 1201525, 1201558, 1201589, 1201620]


# we selected data for calculating the density
mainline_ids_8am = station_data[
    (station_data[0] == pd.Timestamp("2026-01-08 08:00:00")) &
    (station_data[1].isin(mainline_station_ids))
][[0, 1, 5, 9, 10, 11]].copy()


mainline_ids_8am.columns = [
    "timestamp",
    "station_id",
    "station_type",
    "total_flow",
    "avg_occupancy",
    "avg_speed"
]


print(mainline_ids_8am.to_string(index=False))



def get_station_flow(station_id, df):
    rows = df.loc[df["station_id"] == station_id, "total_flow"]
    if rows.empty:
        raise ValueError(f"No total_flow found for station_id {station_id}")
    return rows.iloc[0]

def get_station_speed(station_id, df):
    rows = df.loc[df["station_id"] == station_id, "avg_speed"]
    if rows.empty:
        raise ValueError(f"No avg_speed found for station_id {station_id}")
    return rows.iloc[0]

def station_density(station_id, df):
    flow_5min = get_station_flow(station_id, df)
    speed = get_station_speed(station_id, df)
    flow_hr = flow_5min * 12
    density = flow_hr / speed
    return density

def initial_segment_vehicles(upstream_id, downstream_id, length, df):
    k_up = station_density(upstream_id, df)
    k_down = station_density(downstream_id, df)
    k_avg = (k_up + k_down) / 2
    X_initial = k_avg * length
    return X_initial

X_initial = {}

for segment, stations in segment_stations.items():
    up_id, down_id = stations
    X_initial[segment] = initial_segment_vehicles(
        up_id,
        down_id,
        segment_lengths[segment],
        mainline_ids_8am
    )

for segment in X_initial:

    print(segment, "X_initial =", round(X_initial[segment], 3))

          timestamp  station_id station_type  total_flow  avg_occupancy  avg_speed
2026-01-08 08:00:00     1201419           ML       645.0         0.1582       39.4
2026-01-08 08:00:00     1201469           ML       681.0         0.1384       40.0
2026-01-08 08:00:00     1201497           ML       622.0         0.0525       69.5
2026-01-08 08:00:00     1201525           ML       737.0         0.1677       30.6
2026-01-08 08:00:00     1201558           ML       599.0         0.1914       27.7
2026-01-08 08:00:00     1201589           ML       680.0         0.2410       22.3
2026-01-08 08:00:00     1201620           ML       550.0         0.1686       33.7
Segment 1 X_initial = 228.426
Segment 2 X_initial = 115.327
Segment 3 X_initial = 182.351
Segment 4 X_initial = 82.277
Segment 5 X_initial = 312.707
Segment 6 X_initial = 224.706


In [198]:
#final number of vehicles at the end of 5 min time period

Q_in = {
    "Segment 1": 645,
    "Segment 2": 681,
    "Segment 3": 622,
    "Segment 4": 737,
    "Segment 5": 599,
    "Segment 6": 680

}

u_sum = {
    "Segment 1": 88,
    "Segment 2": 0,
    "Segment 3": 39 + 73,
    "Segment 4": 56,
    "Segment 5": 64,
    "Segment 6": 0
}

Q_out = {
    "Segment 1": 681,
    "Segment 2": 622,
    "Segment 3": 737,
    "Segment 4": 599,
    "Segment 5": 680,
    "Segment 6": 550
}

f_sum = {
    "Segment 1": 0,   # temporary assumption because BRISTOL off-ramp is missing. we will keep using 0 for segment 1 off-ramp for entire process
    "Segment 2": 0,
    "Segment 3": 0,
    "Segment 4": 82,
    "Segment 5": 8,
    "Segment 6": 0
}

X_final = {}
for segment in X_initial:
    X_final[segment] = (
        X_initial[segment] + Q_in[segment] + u_sum[segment]  - Q_out[segment] - f_sum[segment]
    )
for segment in X_final:
    print(segment, "X_final =", round(X_final[segment], 3))



Segment 1 X_final = 280.426
Segment 2 X_final = 174.327
Segment 3 X_final = 179.351
Segment 4 X_final = 194.277
Segment 5 X_final = 287.707
Segment 6 X_final = 354.706


In [199]:

#physical capacity penalty
for lambda_3 in lambda_3_values:
    print("\nlambda_3 =", lambda_3)
    total_physical_penalty = 0

    for segment in X_final:
        overflow = max(0, X_final[segment] - N_max_value[segment])
        penalty = lambda_3 * (overflow ** 2)

        total_physical_penalty += penalty

        print(
            segment,
            "X_final =", round(X_final[segment], 3),
            ", N_max =", round(N_max_value[segment], 3),
            ", overflow =", round(overflow, 3),
            ", penalty =", round(penalty, 3)
        )

    print("Total physical capacity penalty =", round(total_physical_penalty, 3))



lambda_3 = 0
Segment 1 X_final = 280.426 , N_max = 1100.1 , overflow = 0 , penalty = 0
Segment 2 X_final = 174.327 , N_max = 714.1 , overflow = 0 , penalty = 0
Segment 3 X_final = 179.351 , N_max = 1065.36 , overflow = 0 , penalty = 0
Segment 4 X_final = 194.277 , N_max = 289.5 , overflow = 0 , penalty = 0
Segment 5 X_final = 287.707 , N_max = 965.0 , overflow = 0 , penalty = 0
Segment 6 X_final = 354.706 , N_max = 772.0 , overflow = 0 , penalty = 0
Total physical capacity penalty = 0

lambda_3 = 0.5
Segment 1 X_final = 280.426 , N_max = 1100.1 , overflow = 0 , penalty = 0.0
Segment 2 X_final = 174.327 , N_max = 714.1 , overflow = 0 , penalty = 0.0
Segment 3 X_final = 179.351 , N_max = 1065.36 , overflow = 0 , penalty = 0.0
Segment 4 X_final = 194.277 , N_max = 289.5 , overflow = 0 , penalty = 0.0
Segment 5 X_final = 287.707 , N_max = 965.0 , overflow = 0 , penalty = 0.0
Segment 6 X_final = 354.706 , N_max = 772.0 , overflow = 0 , penalty = 0.0
Total physical capacity penalty = 0.0

l

In [200]:
## using actual data , Physical capacity penalty always comes back as 0
## we used fake X_fianl to test if the code works correctly

In [201]:
# synthetic X_final for sanity check
# we added
X_final_fake = {
    "Segment 1": N_max_value["Segment 1"] + 50,
    "Segment 2": N_max_value["Segment 2"] + 60,
    "Segment 3": N_max_value["Segment 3"] + 70,
    "Segment 4": N_max_value["Segment 4"] + 80,
    "Segment 5": N_max_value["Segment 5"] + 90,
    "Segment 6": N_max_value["Segment 6"] + 100
}

for lambda_3 in lambda_3_values:
    print("\nlambda_3 =", lambda_3)
    total_physical_penalty = 0

    for segment in X_final_fake:
        overflow = max(0, X_final_fake[segment] - N_max_value[segment])
        penalty = lambda_3 * (overflow ** 2)

        total_physical_penalty += penalty

        print(
            segment,
            "X_final_fake =", round(X_final_fake[segment], 3),
            ", N_max =", round(N_max_value[segment], 3),
            ", overflow =", round(overflow, 3),
            ", penalty =", round(penalty, 3)
        )
    print("Sum of total physical capacity penalty =", round(total_physical_penalty, 3))







lambda_3 = 0
Segment 1 X_final_fake = 1150.1 , N_max = 1100.1 , overflow = 50.0 , penalty = 0.0
Segment 2 X_final_fake = 774.1 , N_max = 714.1 , overflow = 60.0 , penalty = 0.0
Segment 3 X_final_fake = 1135.36 , N_max = 1065.36 , overflow = 70.0 , penalty = 0.0
Segment 4 X_final_fake = 369.5 , N_max = 289.5 , overflow = 80.0 , penalty = 0.0
Segment 5 X_final_fake = 1055.0 , N_max = 965.0 , overflow = 90.0 , penalty = 0.0
Segment 6 X_final_fake = 872.0 , N_max = 772.0 , overflow = 100.0 , penalty = 0.0
Sum of total physical capacity penalty = 0.0

lambda_3 = 0.5
Segment 1 X_final_fake = 1150.1 , N_max = 1100.1 , overflow = 50.0 , penalty = 1250.0
Segment 2 X_final_fake = 774.1 , N_max = 714.1 , overflow = 60.0 , penalty = 1800.0
Segment 3 X_final_fake = 1135.36 , N_max = 1065.36 , overflow = 70.0 , penalty = 2450.0
Segment 4 X_final_fake = 369.5 , N_max = 289.5 , overflow = 80.0 , penalty = 3200.0
Segment 5 X_final_fake = 1055.0 , N_max = 965.0 , overflow = 90.0 , penalty = 4050.0
Segm

## Safe-threshold penalty calculation
## we used the same syntatic data for final state of the segment (X_final) to varify if the calculation works

In [202]:
# Safe-threshold penalty calculation

lambda_2_values = [0, 0.5, 1, 5]

def safe_threshold_penalty(lambda_2, x_final, x_safe):
    overflow = max(0, x_final - x_safe)
    penalty = lambda_2 * (overflow ** 2)
    return overflow, penalty

for eta_label in safe_occupancy:
    print("\n" + eta_label)

    for lambda_2 in lambda_2_values:
        print("\nlambda_2 =", lambda_2)
        total_safe_penalty = 0

        for segment in X_final:
            overflow, penalty = safe_threshold_penalty(
                lambda_2,
                X_final[segment],
                safe_occupancy[eta_label][segment]
            )

            total_safe_penalty += penalty

            print(
                segment,
                "X_final =", round(X_final[segment], 3),
                ", X_safe =", round(safe_occupancy[eta_label][segment], 3),
                ", overflow =", round(overflow, 3),
                ", penalty =", round(penalty, 3)
            )

        print("Total safe threshold penalty =", round(total_safe_penalty, 3))


eta = 0.3

lambda_2 = 0
Segment 1 X_final = 280.426 , X_safe = 330.03 , overflow = 0 , penalty = 0
Segment 2 X_final = 174.327 , X_safe = 214.23 , overflow = 0 , penalty = 0
Segment 3 X_final = 179.351 , X_safe = 319.608 , overflow = 0 , penalty = 0
Segment 4 X_final = 194.277 , X_safe = 86.85 , overflow = 107.427 , penalty = 0.0
Segment 5 X_final = 287.707 , X_safe = 289.5 , overflow = 0 , penalty = 0
Segment 6 X_final = 354.706 , X_safe = 231.6 , overflow = 123.106 , penalty = 0.0
Total safe threshold penalty = 0.0

lambda_2 = 0.5
Segment 1 X_final = 280.426 , X_safe = 330.03 , overflow = 0 , penalty = 0.0
Segment 2 X_final = 174.327 , X_safe = 214.23 , overflow = 0 , penalty = 0.0
Segment 3 X_final = 179.351 , X_safe = 319.608 , overflow = 0 , penalty = 0.0
Segment 4 X_final = 194.277 , X_safe = 86.85 , overflow = 107.427 , penalty = 5770.294
Segment 5 X_final = 287.707 , X_safe = 289.5 , overflow = 0 , penalty = 0.0
Segment 6 X_final = 354.706 , X_safe = 231.6 , overflow = 123.106

In [203]:
## safe threshold penalty with synthetic X_final

X_final_fake = {
    "Segment 1": N_max_value["Segment 1"] + 50,
    "Segment 2": N_max_value["Segment 2"] + 60,
    "Segment 3": N_max_value["Segment 3"] + 70,
    "Segment 4": N_max_value["Segment 4"] + 80,
    "Segment 5": N_max_value["Segment 5"] + 90,
    "Segment 6": N_max_value["Segment 6"] + 100
}

lambda_2_values = [0, 0.5, 1, 5]

def safe_threshold_penalty(lambda_2, X_final_fake, x_safe):
    overflow = max(0, X_final_fake - x_safe)
    penalty = lambda_2 * (overflow ** 2)
    return overflow, penalty

for eta_label in safe_occupancy:
    print("\n" + eta_label)

    for lambda_2 in lambda_2_values:
        print("\nlambda_2 =", lambda_2)
        total_safe_penalty = 0

        for segment in X_final_fake:
            overflow, penalty = safe_threshold_penalty(
                lambda_2,
                X_final_fake[segment],
                safe_occupancy[eta_label][segment]
            )

            total_safe_penalty += penalty

            print(
                segment,
                "X_final_fake =", round(X_final_fake[segment], 3),
                ", X_safe =", round(safe_occupancy[eta_label][segment], 3),
                ", overflow =", round(overflow, 3),
                ", penalty =", round(penalty, 3)
            )

        print("Total safe threshold penalty =", round(total_safe_penalty, 3))


eta = 0.3

lambda_2 = 0
Segment 1 X_final_fake = 1150.1 , X_safe = 330.03 , overflow = 820.07 , penalty = 0.0
Segment 2 X_final_fake = 774.1 , X_safe = 214.23 , overflow = 559.87 , penalty = 0.0
Segment 3 X_final_fake = 1135.36 , X_safe = 319.608 , overflow = 815.752 , penalty = 0.0
Segment 4 X_final_fake = 369.5 , X_safe = 86.85 , overflow = 282.65 , penalty = 0.0
Segment 5 X_final_fake = 1055.0 , X_safe = 289.5 , overflow = 765.5 , penalty = 0.0
Segment 6 X_final_fake = 872.0 , X_safe = 231.6 , overflow = 640.4 , penalty = 0.0
Total safe threshold penalty = 0.0

lambda_2 = 0.5
Segment 1 X_final_fake = 1150.1 , X_safe = 330.03 , overflow = 820.07 , penalty = 336257.402
Segment 2 X_final_fake = 774.1 , X_safe = 214.23 , overflow = 559.87 , penalty = 156727.208
Segment 3 X_final_fake = 1135.36 , X_safe = 319.608 , overflow = 815.752 , penalty = 332725.663
Segment 4 X_final_fake = 369.5 , X_safe = 86.85 , overflow = 282.65 , penalty = 39945.511
Segment 5 X_final_fake = 1055.0 , X_safe =

# Above calculations are all done with the dataset given from PeMS

